# MACHO import

Author: Konstantin Malanchev

Last run: 2026-09-16

https://macho.nci.org.au/

Imports the per-field parquet files produced by `macho.raw.ipynb`.

In [1]:
import hats_import
import pyarrow as pa
import pyarrow.parquet
from dask.distributed import Client
from hats.pixel_math.spatial_index import SPATIAL_INDEX_COLUMN
from hats_import import CollectionArguments, VerificationArguments, pipeline, pipeline_with_client
from upath import UPath

hats_import.__version__

/astro/users/kmalanch/.virtualenvs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'0.10.4'

In [2]:
parquet_dir = UPath("/astro/store/shire/hats/raw/macho_parquet")
output_path = UPath("/astro/store/shire/hats/catalogs")

In [3]:
parquet_files = sorted(parquet_dir.glob("macho_*/*.parquet"))
total_rows = sum(pa.parquet.read_metadata(path).num_rows for path in parquet_files)
len(parquet_files), total_rows

(25667, 69173447)

In [4]:
# byte_stream_split for the base float columns, dictionary for everything else.
# https://github.com/astronomy-commons/hats/issues/742
parquet_schema = pa.parquet.ParquetFile(parquet_files[0]).schema
leaf_columns = [parquet_schema.column(index) for index in range(len(parquet_schema))]
base_float_columns = [
    column.path
    for column in leaf_columns
    if column.physical_type in ("FLOAT", "DOUBLE")
    and not column.path.startswith("lc.")
    and column.path not in ("ra", "dec")
]
dictionary_columns = [
    column.path for column in leaf_columns if column.path not in base_float_columns
] + [SPATIAL_INDEX_COLUMN]

In [5]:
args = (
    CollectionArguments(
        completion_email_address="kmalanch@andrew.cmu.edu",
        output_artifact_name="macho",
        output_path=output_path,
        progress_bar=True,
        simple_progress_bar=True,
        write_table_kwargs={
            "write_page_index": True,
            "data_page_size": 128 * 1024,
            "compression": "zstd",
            "compression_level": 15,
            "use_byte_stream_split": base_float_columns,
            "use_dictionary": dictionary_columns,            
        },
    )
    .catalog(
        output_artifact_name="macho",
        input_file_list=parquet_files,
        file_reader="parquet",
        ra_column="ra",
        dec_column="dec",
        sort_columns="starid",
        expected_total_rows=total_rows,
        # Not availble yet
        # byte_pixel_threshold=256 * 1024 * 1024,
        pixel_threshold=50_000,
        highest_healpix_order=10,
        skymap_alt_orders=[2, 4, 6],
    )
    .add_margin(margin_threshold=10.0, is_default=True)
)

In [6]:
with Client(n_workers=4, threads_per_worker=1, memory_limit="32GB") as client:
    pipeline_with_client(args, client)

Catalog: Planning  :  25%|██▌       | 1/4 [00:00<00:00,  5.67it/s]

tmp_path (/astro/store/shire/hats/catalogs/macho/intermediate/macho/intermediate) contains intermediate files; resuming prior progress.


Catalog: Reducing  :  13%|█▎        | 415/3311 [37:33<1:32:06,  1.91s/it]2026-09-19 13:03:46,717 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 23.92 GiB -- Worker memory limit: 29.80 GiB
2026-09-19 13:03:47,845 - distributed.worker.memory - WARNING - Worker is at 77% memory usage. Resuming worker. Process memory: 23.05 GiB -- Worker memory limit: 29.80 GiB
Catalog: Reducing  :  16%|█▌        | 519/3311 [46:16<4:46:16,  6.15s/it]2026-09-19 13:12:33,849 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 21.00 GiB -- Worker memory limit: 29.80 GiB
2026-09-19 13:12:36,779 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process memory: 23.92 GiB -- Worker memory 

In [7]:
# Base floats should be BYTE_STREAM_SPLIT, the rest RLE_DICTIONARY.
catalog_file = next((output_path / "macho" / "macho" / "dataset").rglob("*.parquet"))
metadata = pa.parquet.read_metadata(catalog_file)
{
    metadata.row_group(0).column(index).path_in_schema: metadata.row_group(0).column(index).encodings
    for index in range(metadata.num_columns)
    if metadata.row_group(0).column(index).path_in_schema
    in ("rmagave", "kv", "ra", "starid", "lc.rmag.list.element", SPATIAL_INDEX_COLUMN)
}

{'_healpix_29': ('PLAIN', 'RLE', 'RLE_DICTIONARY'),
 'starid': ('PLAIN', 'RLE', 'RLE_DICTIONARY'),
 'rmagave': ('RLE', 'BYTE_STREAM_SPLIT'),
 'ra': ('PLAIN', 'RLE', 'RLE_DICTIONARY'),
 'kv': ('RLE', 'BYTE_STREAM_SPLIT'),
 'lc.rmag.list.element': ('PLAIN', 'RLE', 'RLE_DICTIONARY')}

In [8]:
args = VerificationArguments(
    input_catalog_path=output_path / "macho",
    output_path="./verification/macho",
)
pipeline(args)

Loading dataset and schema.

Starting: Test hats.io.validation.is_valid_collection.
Result: PASSED

Starting: Test that files in _metadata match the data files on disk.
Result: PASSED

Starting: Test that number of rows are equal.
	file footers vs catalog properties
	file footers vs _metadata
Result: PASSED

Starting: Test that schemas are equal, excluding metadata.
	_common_metadata vs truth
	_metadata vs truth
	file footers vs truth
Result: PASSED

Verifier results written to verification/macho/verifier_results.csv
Elapsed time (seconds): 568.48
